In [2]:
import os
import pandas as pd
from sqlalchemy import create_engine
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import mlflow
import numpy as np

REFERENCE_BATCH = "group-5-batch-4"
CURRENT_BATCH = "group-5-batch-5"
SAMPLE_SIZE = 10_000

db_url = (
    f"postgresql+psycopg2://{os.environ['POSTGRES_USER']}:"
    f"{os.environ['POSTGRES_PASSWORD']}@postgres:5432/"
    f"{os.environ['POSTGRES_DB']}"
)
engine = create_engine(db_url)

def load_batch(batch_id):
    return pd.read_sql(
        """
        SELECT *
        FROM gold.property_training_features
        WHERE batch_id = %(batch_id)s
        LIMIT %(limit)s
        """,
        engine,
        params={"batch_id": batch_id, "limit": SAMPLE_SIZE},
    )

reference_df = load_batch(REFERENCE_BATCH)
current_df = load_batch(CURRENT_BATCH)

In [3]:
mlflow.set_tracking_uri("http://mlflow:5000")
client = mlflow.tracking.MlflowClient()

champion = client.get_model_version_by_alias(
    "property_price_model",
    "champion",
)

training_run = client.get_run(champion.run_id)
training_mae = training_run.data.metrics["test_mae"]

model = mlflow.pyfunc.load_model("models:/property_price_model@champion")

MODEL_FEATURES = [
    "brokered_by", "bed", "bath", "acre_lot", "street",
    "zip_code", "house_size", "prev_sold_year",
    "status", "city", "state",
]

def evaluate_batch(data, name):
    predictions = model.predict(data[MODEL_FEATURES])
    actual = data["target_price"]

    return {
        "dataset": name,
        "mae": mean_absolute_error(actual, predictions),
        "rmse": np.sqrt(mean_squared_error(actual, predictions)),
        "r2": r2_score(actual, predictions),
    }

concept_drift_results = pd.DataFrame([
    {
        "dataset": "training_test_baseline",
        "mae": training_mae,
        "rmse": np.nan,
        "r2": np.nan,
    },
    evaluate_batch(reference_df, REFERENCE_BATCH),
    evaluate_batch(current_df, CURRENT_BATCH),
])

concept_drift_results

,dataset,mae,rmse,r2
0,training_test_baseline,41027.940826,NaN,NaN
1,group-5-batch-4,40677.616813,4.877413e+04,0.283242
2,group-5-batch-5,941216.054574,2.159470e+06,-0.230809


In [5]:
mlflow.set_experiment("property_price_regression")

<Experiment: artifact_location='s3://mlflow/3', creation_time=1794786614437, experiment_id='3', last_update_time=1794786614437, lifecycle_stage='active', name='property_price_regression', tags={}>

In [6]:
import json
from datetime import datetime, timezone

current_mae = float(
    concept_drift_results.loc[
        concept_drift_results["dataset"] == CURRENT_BATCH, "mae"
    ].iloc[0]
)

mae_multiplier = current_mae / float(training_mae)

decision = {
    "assessed_at": datetime.now(timezone.utc).isoformat(),
    "reference_batch": REFERENCE_BATCH,
    "current_batch": CURRENT_BATCH,
    "training_baseline_mae": float(training_mae),
    "current_batch_mae": current_mae,
    "mae_multiplier": mae_multiplier,
    "decision": "review_data_quality_and_train_candidate",
    "promotion": "not_automatic",
}

concept_drift_results.to_csv(
    "reports/concept_drift_results.csv",
    index=False,
)

with open("reports/concept_drift_decision.json", "w") as file:
    json.dump(decision, file, indent=2)

with mlflow.start_run(run_name="concept_drift_assessment_batch_5") as run:
    mlflow.log_params({
        "reference_batch": REFERENCE_BATCH,
        "current_batch": CURRENT_BATCH,
        "decision": decision["decision"],
        "promotion": decision["promotion"],
    })
    mlflow.log_metrics({
        "training_baseline_mae": float(training_mae),
        "current_batch_mae": current_mae,
        "mae_multiplier": mae_multiplier,
    })
    mlflow.log_artifact("reports/concept_drift_results.csv", artifact_path="drift")
    mlflow.log_artifact("reports/concept_drift_decision.json", artifact_path="drift")

print(decision)


2026/11/22 16:36:19 INFO mlflow.tracking._tracking_service.client: 🏃 View run concept_drift_assessment_batch_5 at: http://mlflow:5000/#/experiments/3/runs/4cf78d98c8d349269a6e0eed907552e8.
2026/11/22 16:36:19 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://mlflow:5000/#/experiments/3.


{'assessed_at': '2026-11-22T16:36:19.190955+00:00', 'reference_batch': 'group-5-batch-4', 'current_batch': 'group-5-batch-5', 'training_baseline_mae': 41027.94082558441, 'current_batch_mae': 941216.0545742317, 'mae_multiplier': 22.940855320413824, 'decision': 'review_data_quality_and_train_candidate', 'promotion': 'not_automatic'}
